# Week 6 — Multi-table Joins and Complex Aggregations: Geographic Revenue Analysis
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Combine customer geography, payments, and delivery timing into a single full-pipeline query across `orders` → `customers` → `order_payments`
- Compute a date difference on SQLite's TEXT date columns using `julianday()`, and read the result as a business number (days in transit)
- Recognise which table in a multi-table chain is the *safe base* and which one fans out, and know exactly which of your aggregates that fan-out corrupts
- Rank and filter grouped geographic results with `ORDER BY`, `LIMIT`, and `HAVING`, ready to extend the same chain to 4 and 5 tables in today's exercises

Run the setup cell first. It loads all 8 Olist tables into a SQLite database and connects the
`%%sql` magic to it. Everything below this cell depends on it — if a query errors with
*"no such table"*, come back and run this one again.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71

## Why this matters

Brazil is not one market. It is **27 states** spread across a country bigger than the continental
United States, and Olist sells into all of them. Yesterday you learned to chain three tables to
answer *what* sells. Today's question is *where* — and it is the one an operations director
actually loses sleep over.

The problem is that no single table can answer it. The money lives in `order_payments`. The
delivery dates live in `orders`. And the **state** — the one column the whole question turns on —
lives in neither: it sits in `customers`, as `customer_state`. To put revenue, geography, and
delivery speed on the same row you have to walk the full pipeline: `orders` → `customers` →
`order_payments`.

Here is why it is worth the walk. Of the 96,478 delivered orders in this database, **40,501 of
them — 42% — went to customers in São Paulo state alone.** One state out of twenty-seven takes
four orders in ten. Whether that concentration is a strength or a fragility depends on numbers you
are about to compute: what those orders are worth, and how long they take to arrive compared with
everywhere else. By the end of this notebook you will have a single query that answers all of it,
state by state.

## 1. Building the pipeline — step one: `orders` → `customers`

Every multi-table query is easier to write if you build it **one join at a time and look at the
result after each step**, rather than typing all four tables and hoping. Today's pipeline has two
joins, so we will add them one at a time.

The first link answers "where did this order go?". `orders` carries a `customer_id`, and
`customers` carries that same `customer_id` plus the geography columns — `customer_city`,
`customer_state`, `customer_zip_code_prefix`. That shared column is the bridge, exactly as in the
two-table joins from Week 3.

One thing to be careful about, because it bites people every cohort: `customers` has **two**
id columns. `customer_id` is unique per *order* — it is the key you join on. `customer_unique_id`
is the key that identifies a real person across repeat purchases. Join on the wrong one and your
row counts will be quietly wrong. The query below joins correctly and just shows five rows, so you
can see what the joined grain actually looks like before any aggregation happens.

In [ ]:
%%sql
-- Step 1 of the pipeline: attach each order to the customer who placed it.
-- No aggregation yet — just look at the joined rows.
SELECT o.order_id,
       o.order_status,
       c.customer_city,
       c.customer_state
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
LIMIT 5

Each row now carries the order's identity and status from `orders` and the customer's location from
`customers` — one row per order, because both tables hold exactly one row per order (99,441 each).
That is important, and we will come back to it: this two-table join is at **order grain**, so it is
completely safe to count and aggregate over.

Now aggregate it. `GROUP BY c.customer_state` collapses those joined rows into one row per state,
`WHERE o.order_status = 'delivered'` restricts us to orders that actually arrived (no point
measuring delivery speed on an order that was cancelled), and `ORDER BY ... DESC LIMIT 8` gives us
the eight biggest states.

In [ ]:
%%sql
-- Step 2: aggregate the two-table join — how many delivered orders per state?
SELECT c.customer_state,
       COUNT(DISTINCT o.order_id) AS order_count
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY order_count DESC
LIMIT 8
-- Expected top row: SP with 40,501 delivered orders

**Expected output — delivered orders by state (top 8):**

| customer_state | order_count |
|---|---|
| SP | 40,501 |
| RJ | 12,350 |
| MG | 11,354 |
| RS | 5,345 |
| PR | 4,923 |
| SC | 3,546 |
| BA | 3,256 |
| DF | 2,080 |

São Paulo (40,501) has more delivered orders than the next **five** states put together. Rio de
Janeiro and Minas Gerais follow at roughly a quarter of SP's volume each, and by the eighth state —
Distrito Federal, the capital district — we are down to 2,080 orders. This is the shape of Olist's
market, and it is the backdrop for everything else today.

But order *count* is not order *value*, and it says nothing about whether those orders arrive
quickly. For that we need the third table.

## 2. The full pipeline — adding payments and delivery time

The second join brings in `order_payments`, which holds `payment_value` — what the customer
actually paid, including freight and instalment charges. It links to `orders` on `order_id`.

That gives us money. For delivery speed we need a *date difference*, and here SQLite has a quirk
worth knowing well. SQLite has **no date type**. `order_purchase_timestamp` and
`order_delivered_customer_date` are plain TEXT strings like `'2017-10-02 10:56:33'`. You cannot
subtract two strings and get a number of days.

The fix is **`julianday()`**, which converts a date string into a single number: days elapsed since
a fixed astronomical reference point in 4713 BC. The absolute value is meaningless to us — but the
*difference* between two julian day numbers is exactly the number of days between them, decimals
and all. So `julianday(delivered) - julianday(purchased)` is the transit time in days, and
`AVG(...)` over that is the state's average delivery time.

Put the three tables and the date arithmetic together and you get the query the whole session has
been building towards.

In [ ]:
%%sql
-- THE FULL PIPELINE: geography (customers) + money (order_payments) + timing (orders),
-- restricted to orders that were actually delivered, one row per state.
SELECT c.customer_state,
       COUNT(DISTINCT o.order_id)      AS order_count,
       ROUND(SUM(op.payment_value), 2) AS total_payment_value,
       ROUND(AVG(op.payment_value), 2) AS avg_order_value,
       ROUND(AVG(
           julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)
       ), 1)                           AS avg_delivery_days
FROM orders o
JOIN customers c       ON o.customer_id = c.customer_id
JOIN order_payments op ON o.order_id = op.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_payment_value DESC
LIMIT 8
-- Expected top row: SP | 40,500 orders | R$5,770,266.19 | R$136.39 | 8.8 days

**Expected output — the full pipeline, top 8 states by payment value:**

| customer_state | order_count | total_payment_value | avg_order_value | avg_delivery_days |
|---|---|---|---|---|
| SP | 40,500 | 5,770,266.19 | 136.39 | 8.8 |
| RJ | 12,350 | 2,055,690.45 | 158.08 | 15.4 |
| MG | 11,354 | 1,819,277.61 | 154.12 | 12.0 |
| RS | 5,345 | 861,802.40 | 155.45 | 15.3 |
| PR | 4,923 | 781,919.55 | 152.45 | 12.0 |
| SC | 3,546 | 595,208.40 | 162.58 | 14.9 |
| BA | 3,256 | 591,270.60 | 169.76 | 19.3 |
| DF | 2,080 | 346,146.17 | 161.60 | 12.9 |

Read it as a business person. Three findings jump out, and none of them were visible in the
order-count table:

1. **São Paulo is cheap and fast.** It has the *lowest* average order value of the eight
   (R$136.39) and by far the fastest delivery (8.8 days — the next best is 12.0). Olist's sellers
   are concentrated in SP, so SP customers are mostly buying from sellers in their own state.
   Short distance, low freight, quick arrival.
2. **Distance costs money and time.** Bahia (BA), up the north-east coast, waits **19.3 days** —
   more than double SP — and pays the highest average order value of the eight at R$169.76. Part of
   that R$169.76 is freight the customer is absorbing to get goods shipped a long way.
3. **Value and volume rank differently.** DF has fewer orders than SC but similar economics; RJ is
   worth R$2.06m at 15.4 days, slow for a state that is only 400 km from São Paulo.

One more thing to notice, and it is not a typo: **SP shows 40,500 here but 40,501 in the previous
query.** One SP order vanished when we added the payments join. We find it in the next section —
that kind of silent single-row loss is exactly what multi-table joins do to you if you are not
watching.

---
## 🤖 Using DeepSeek this week

A five-table pipeline is a lot of typing, and DeepSeek is genuinely good at drafting the join
chain if you describe the tables and keys clearly. The rule from Week 4 has not changed:
**draft, run, verify, then trust.**

Geographic queries have one AI failure mode that deserves naming. Ask for "revenue by customer
state" and a model will very often write `JOIN customers c ON o.customer_id = c.customer_unique_id`
— the wrong id column, because `customer_unique_id` *sounds* more authoritative. That query does
not error. It returns a smaller, plausible-looking table of states and totals that is simply wrong.
Nothing about the output tells you.

The protocol:
1. **Ask** DeepSeek precisely, naming your keys: *"Using SQLite, total payment value for delivered
   orders from customers in São Paulo state. Tables: orders (order_id, customer_id, order_status),
   customers (customer_id, customer_state), order_payments (order_id, payment_value). Join orders to
   customers on customer_id and orders to order_payments on order_id."*
2. **Run** what it gives you, in a `%%sql` cell, against this database.
3. **Verify** against a number you already computed. You know SP's delivered payment total from the
   table above: **R$5,770,266.19**. If the AI's query returns anything else, its query is wrong —
   not the data. Only after it reproduces a number you have already verified should you trust it on
   a question you haven't solved yourself.

The cell below is step 3 — the anchor value, computed independently, that any AI-drafted attempt
must match.

In [ ]:
%%sql
-- Verification anchor for AI-drafted SQL: a value we have already confirmed above.
SELECT ROUND(SUM(op.payment_value), 2) AS sp_delivered_payment_total
FROM orders o
JOIN customers c       ON o.customer_id = c.customer_id
JOIN order_payments op ON o.order_id = op.order_id
WHERE o.order_status = 'delivered'
  AND c.customer_state = 'SP'          -- Expected: 5,770,266.19

## Going deeper — the fan-out that survives `SUM` but ruins `AVG`

Yesterday's rule was: `order_items`, `order_payments`, and `order_reviews` each hold **many rows per
`order_id`**, so joining them multiplies rows. Today's pipeline touches exactly one of them —
`order_payments` — and the chain `orders` → `customers` is at clean order grain, so this query is
fan-out-*safe as written*. But "safe" does not mean "unaffected", and this is the nuance worth
carrying into Week 7.

`order_payments` has 103,886 rows describing 99,440 orders: an order paid in three instalments, or
part-paid with a voucher, contributes several rows. Now look at what that does to each aggregate:

- **`SUM(op.payment_value)` is fine.** Adding up every payment row for a state gives the same total
  as adding up each order's combined payment. Splitting a R$300 order into three R$100 rows changes
  nothing about the sum.
- **`COUNT(DISTINCT o.order_id)` is fine** — that is *why* it is written with `DISTINCT`. Plain
  `COUNT(*)` would count payment rows, not orders.
- **`AVG(op.payment_value)` is NOT what its name suggests.** It is the average *payment row*, not
  the average *order*. Every instalment is a separate small row dragging the mean down, and states
  where instalment payment is common get dragged down further than others.

The way to see the size of that effect is to compute the same table the fan-out-safe way: collapse
`order_payments` to exactly one row per `order_id` in a `WITH` CTE **first**, then join that.

In [ ]:
%%sql
-- The fan-out-SAFE pattern: collapse order_payments to 1 row per order_id in a CTE,
-- THEN join. Now every order contributes exactly one payment row.
WITH pay AS (
    SELECT order_id,
           SUM(payment_value) AS order_payment
    FROM order_payments
    GROUP BY order_id
)
SELECT c.customer_state,
       COUNT(DISTINCT o.order_id)        AS order_count,
       ROUND(SUM(pay.order_payment), 2)  AS total_payment_value,
       ROUND(AVG(pay.order_payment), 2)  AS avg_order_value,
       ROUND(AVG(
           julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)
       ), 1)                             AS avg_delivery_days
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN pay         ON o.order_id = pay.order_id     -- 1:1 now — no fan-out
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY total_payment_value DESC
LIMIT 8
-- Expected top row: SP | 40,500 orders | R$5,770,266.19 | R$142.48 | 8.8 days

**Expected output — the same eight states, computed at true order grain:**

| customer_state | order_count | total_payment_value | avg_order_value | avg_delivery_days |
|---|---|---|---|---|
| SP | 40,500 | 5,770,266.19 | 142.48 | 8.8 |
| RJ | 12,350 | 2,055,690.45 | 166.45 | 15.3 |
| MG | 11,354 | 1,819,277.61 | 160.23 | 12.0 |
| RS | 5,345 | 861,802.40 | 161.24 | 15.3 |
| PR | 4,923 | 781,919.55 | 158.83 | 12.0 |
| SC | 3,546 | 595,208.40 | 167.85 | 15.0 |
| BA | 3,256 | 591,270.60 | 181.59 | 19.3 |
| DF | 2,080 | 346,146.17 | 166.42 | 13.0 |

Compare the two tables column by column — this is the whole lesson in one picture:

- `total_payment_value` is **identical to the cent** in both. `SUM` genuinely survives the fan-out.
- `order_count` is identical. `COUNT(DISTINCT ...)` did its job.
- `avg_order_value` **moved in every single row**: SP R$136.39 → **R$142.48**, BA R$169.76 →
  **R$181.59**. The first table's averages were per-payment-row and understated the true average
  order by R$6–12 depending on the state.
- `avg_delivery_days` shifted slightly too (SC 14.9 → 15.0, DF 12.9 → 13.0), because in the first
  query a three-instalment order counted its delivery time three times.

So: when someone hands you a joined aggregate, do not ask "does it run?" — ask **"what is one row
of this join, and does that match what my aggregate claims to measure?"** That question is the
whole of Week 7.

### And the order that disappeared

Back in the full-pipeline result, SP dropped from 40,501 to 40,500. A plain `JOIN` (an
**inner** join) keeps only rows that match on *both* sides — so any delivered order with no row at
all in `order_payments` is silently dropped from the result. It does not warn you. The state total
just comes out one order light.

`order_payments` covers 99,440 of the 99,441 orders, so exactly one order is unmatched. The query
below finds it: `LEFT JOIN` keeps every delivered order whether or not it has a payment, and
`WHERE op.order_id IS NULL` keeps only the ones that didn't match. Note `IS NULL`, never `= NULL` —
`= NULL` is never true for anything in SQL.

In [ ]:
%%sql
-- Which delivered order has no payment row at all? LEFT JOIN + IS NULL finds the unmatched rows.
SELECT o.order_id,
       c.customer_state,
       o.order_status
FROM orders o
JOIN customers c            ON o.customer_id = c.customer_id
LEFT JOIN order_payments op ON o.order_id = op.order_id
WHERE o.order_status = 'delivered'
  AND op.order_id IS NULL
-- Expected: exactly 1 row — order bfbd0f9bdef84302105ad712db648a6c, customer_state SP

One row, in SP — that is our missing order, `bfbd0f9bdef84302105ad712db648a6c`. On a table of
40,500 it changes nothing. On a smaller slice it could change everything, and the point is that
**you only ever find these by looking.** Whenever a count drops after you add a join, that join is
an inner join throwing rows away, and you should be able to say exactly which rows and why.

### Ranking on time instead of money, with `HAVING`

`ORDER BY total_payment_value DESC` answers "where is the money?". Swap the sort column and the
same pipeline answers a completely different operational question: **"where are we slowest?"**

There is a trap in doing that naively. Sort all 27 states by average delivery days and the top of
the list fills up with tiny states where a handful of unlucky orders swing the average wildly. The
fix is `HAVING`, which filters **groups** after aggregation the way `WHERE` filters **rows** before
it. `HAVING COUNT(DISTINCT o.order_id) >= 300` keeps only states with enough orders for the average
to mean something. You cannot write that condition in `WHERE` — at `WHERE` time the groups don't
exist yet.

Because this question is only about timing, we can drop `order_payments` entirely and run on the
clean order-grain `orders` → `customers` join. Fewer tables, no fan-out to reason about at all.

In [ ]:
%%sql
-- Slowest states by average delivery time, restricted to states with enough orders to be meaningful.
SELECT c.customer_state,
       COUNT(DISTINCT o.order_id) AS order_count,
       ROUND(AVG(
           julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)
       ), 1) AS avg_delivery_days
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
HAVING COUNT(DISTINCT o.order_id) >= 300      -- filters GROUPS, after aggregation
ORDER BY avg_delivery_days DESC
LIMIT 8
-- Expected top row: AL with 397 orders and 24.5 average delivery days

**Expected output — the eight slowest states (min. 300 delivered orders):**

| customer_state | order_count | avg_delivery_days |
|---|---|---|
| AL | 397 | 24.5 |
| PA | 946 | 23.8 |
| MA | 717 | 21.6 |
| SE | 335 | 21.5 |
| CE | 1,279 | 21.3 |
| PB | 517 | 20.4 |
| PI | 476 | 19.5 |
| RN | 474 | 19.3 |

Every state on this list is in Brazil's north or north-east — and **not one of them appeared in the
revenue top 8.** Alagoas customers wait 24.5 days on average against São Paulo's 8.8: nearly three
times as long, from the same marketplace. That is the geography of Olist's logistics problem stated
in one query, and it is precisely the kind of finding that only surfaces once you have joined
customer location to order timing.

## Common mistakes

**Mistake 1 — `COUNT(*)` after joining `order_payments`.** Once payments are joined, one row is one
*payment*, not one order. For SP the join produces **42,308 rows** describing **40,500 orders** —
report the first number as "orders" and you have overstated São Paulo by 1,808. It never errors and
the number looks completely reasonable, which is what makes it dangerous.

**Mistake 2 — joining `customers` on `customer_unique_id`.** `customers` has both `customer_id`
(one per order — the join key) and `customer_unique_id` (one per real person). Writing
`ON o.customer_id = c.customer_unique_id` returns zero rows, and an empty result is easy to
misread as "no data for this filter" rather than "my join is wrong".

**Mistake 3 — subtracting the date strings directly.** `order_delivered_customer_date -
order_purchase_timestamp` does not error in SQLite. It silently coerces both TEXT values to numbers
— `'2017-10-02 10:56:33'` becomes `2017` — and returns a difference of years-as-integers, usually
`0`. You get a column full of zeros and no warning at all. Always wrap both sides in `julianday()`.

**Mistake 4 — putting an aggregate condition in `WHERE`.** `WHERE COUNT(DISTINCT o.order_id) >= 300`
raises *"misuse of aggregate function COUNT()"*. Aggregates are only available after grouping, so
that condition belongs in `HAVING`.

The cell below shows the first mistake as a comment and then runs the correct comparison live, so
you can see both numbers side by side and the exact size of the gap.

In [ ]:
%%sql
-- ── COMMON MISTAKE ──────────────────────────────────────────────────
-- WRONG — after joining order_payments, one row is one PAYMENT, not one order:
--   SELECT c.customer_state, COUNT(*) AS order_count ...
-- WRONG — the wrong customer key returns zero rows, silently:
--   JOIN customers c ON o.customer_id = c.customer_unique_id
-- WRONG — TEXT dates cannot be subtracted directly (returns 0, no error):
--   AVG(o.order_delivered_customer_date - o.order_purchase_timestamp)
-- CORRECT — both counts shown together so you can see the fan-out gap:
SELECT c.customer_state,
       COUNT(*)                   AS joined_payment_rows,   -- Expected: 42,308 (WRONG as an order count)
       COUNT(DISTINCT o.order_id) AS order_count            -- Expected: 40,500 (CORRECT)
FROM orders o
JOIN customers c       ON o.customer_id = c.customer_id
JOIN order_payments op ON o.order_id = op.order_id
WHERE o.order_status = 'delivered'
  AND c.customer_state = 'SP'
GROUP BY c.customer_state

## Group Exercise — plan the chain before you write it

⏱ ~8 min · pairs or threes · pen and paper only, no code yet

Today's exercises notebook asks you to push the pipeline from three tables to **four and five**.
Before you touch a keyboard, plan the chains together. For each question below, agree on (a) the
list of tables in order, (b) the key each `JOIN ... ON ...` uses, and (c) whether any table in your
chain fans out — and if so, what you will do about it.

1. **Freight from SP sellers.** Average freight paid by customers in each state, when ordering from
   sellers based in SP. Chain: `orders` → `customers` → `order_items` → `sellers`. Where does
   `freight_value` live, and where does `seller_state` live? Which of the two are you filtering on
   and which are you grouping by?
2. **SP vs RJ category taste.** Top 5 product categories in SP compared with RJ. Chain:
   `orders` → `customers` → `order_items` → `products` → `product_category_translation` — five
   tables, four joins. Trace the keys end to end and say them out loud.
3. **GMV per seller state.** Total GMV (`price + freight_value`) grouped by the *seller's* state.
   This one is shorter than it looks — which two tables do you actually need?
4. **Premium categories.** Categories whose average item price exceeds R$150. Which clause holds
   the `> 150` condition, and why can it not go in `WHERE`?

Nominate one person per group to report back on question 2 — it is the longest chain you have
built so far, and saying it aloud is the fastest way to find the gap in it.

*(You will actually write and run these in `week-06-thu-exercises.ipynb`; this is the planning
pass.)*

## Mini-challenge — your turn

⏱ ~5–10 min · individual

Take the full-pipeline query from section 2 and point it at a **single state** instead of ranking
the top eight. Write one query returning `order_count`, `total_payment_value`, `avg_order_value`,
and `avg_delivery_days` for **Rio de Janeiro (`'RJ'`) only**.

Two hints:
- Keep all three tables and both `JOIN ... ON ...` lines exactly as they are. The chain does not
  change — only the filter does.
- Add `AND c.customer_state = 'RJ'` to the existing `WHERE`, and drop the `ORDER BY` / `LIMIT`
  (there is only one row to return). Keep the `GROUP BY`.

**Expected:** one row — 12,350 orders, R$2,055,690.45 total payment value, R$158.08 average order
value, 15.4 average delivery days. Check it against the full-pipeline table above: your single row
should match the RJ line exactly.

**Stretch, if you finish early:** run it again with the `WITH pay AS (...)` CTE from the Going
Deeper section in place of the direct `order_payments` join. Which of the four numbers change, and
which do not? *(Answer: the total stays at R$2,055,690.45 and the count stays at 12,350, but
`avg_order_value` rises to R$166.45 and `avg_delivery_days` settles to 15.3 — exactly the fan-out
effect described above.)*

In [ ]:
%%sql
-- ⏱ ~5-10 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo

## Session Summary

| Pattern | What it does | Example |
|---|---|---|
| Full-pipeline `JOIN` chain | walks `orders` → `customers` → `order_payments` so geography, money, and timing land on one row | `FROM orders o JOIN customers c ON o.customer_id = c.customer_id JOIN order_payments op ON o.order_id = op.order_id` |
| `julianday()` difference | turns two TEXT date columns into a number of days | `AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp))` |
| `COUNT(DISTINCT ...)` | counts real orders, not fanned-out payment rows | `COUNT(DISTINCT o.order_id) AS order_count` |
| Pre-aggregating `WITH` CTE | collapses `order_payments` to 1 row per order so `AVG` means "average order" | `WITH pay AS (SELECT order_id, SUM(payment_value) AS order_payment FROM order_payments GROUP BY order_id)` |
| `HAVING` | filters *groups* after aggregation, where `WHERE` cannot reach | `HAVING COUNT(DISTINCT o.order_id) >= 300` |
| `LEFT JOIN` + `IS NULL` | finds the rows an inner join would silently discard | `LEFT JOIN order_payments op ON o.order_id = op.order_id WHERE op.order_id IS NULL` |

**What you can now answer that you couldn't yesterday:** where Olist's revenue actually comes from
(SP alone: R$5,770,266.19 across 40,500 delivered orders), how long customers in each state wait
(8.8 days in SP against 24.5 in AL), and — just as important — *which of your own aggregates you
should not believe* once a fan-out table is in the join.

---
**Coming up Wednesday (Week 7)**: **CTEs and advanced analytics** — you met `WITH ... AS (...)`
today as a fan-out fix, and next week it becomes the main tool: breaking a long, unreadable query
into named, testable steps you can build and check one at a time.